# TableTrace GPU 보강 실험 (2번 노트북)

**이 노트북은 Kaggle(T4 ×2 권장) 또는 Colab에서 실행합니다. 작성자가 실행해 보지 못했습니다**(개발 컨테이너에 GPU가 없음). CPU에서 확인한 것은 각 스크립트의 작은 무작위 모델 시험뿐입니다(`tests/test_lora_sft_smoke.py`, `tests/test_exaone_grpo_smoke.py`). 처음 도는 설정이므로 실패하면 **그 셀의 출력 전체**를 알려 주세요.

1번 노트북(`gpu_experiments.ipynb`)의 결과는 저장소에 이미 있습니다. 이 노트북은 공고와 대조했을 때 비어 있던 부분을 채우는 실험입니다. **0번 절은 반드시 실행**하고, A~H는 서로 독립이라 골라서 하면 됩니다. 무료 GPU 주 30시간 안에서 권하는 순서는 **H → A → B(7000) → F → G → E → C → A2 → D**입니다. 앞의 것일수록 실패할 일이 적고 결과의 쓸모가 큽니다. A2는 A절의 변수를 쓰므로 A 다음에 실행하세요.

| 절 | 무엇을 재나 | GPU | 예상 시간 | 필요한 것 |
|---|---|---|---|---|
| A | 큰 EXAONE(7.8B, 32B)을 튜닝 없이 쓰면 근거 셀이 맞는가 | 7.8B는 1장, 32B는 2장 | 40분 + 60분 | 없음 |
| A2 | 모델이 도구를 스스로 골라 표를 읽는 방식(Tool Calling) | 1장(32B는 2장) | 30분 + 60분 | A절을 먼저 실행(`TOK` 변수) |
| B | 학습 데이터를 3.6천 → 1만 → 3.4만으로 늘리면 | 1장(2장이면 절반) | 1.5시간, 5시간 | 없음 |
| C | 멀티 GPU 학습(DDP) 속도 | 2장 | 15분 | 없음 |
| D | 강화학습(GRPO)이 근거 셀 오류를 줄이는가 | 1장 | 1.5~2시간 | **본인이 쓴 `training/evidence_reward.py`, `training/grpo.py`** |
| E | 4비트 양자화의 정확도·속도·메모리 | 1장 | 20분 | 1번 노트북 또는 B의 병합 모델 |
| F | 열 번호 형식: 전체 데이터, 시드 2개 더 | 1장 | 1시간 | 없음 |
| G | 영어 화학 문서에서 EXAONE | 1장 | 30분 | 없음 |
| H | 큰 표(공개 dev, 학습에 없던 표 3,072문항): 행 선별 | 1장 | 20분 | 1번 노트북 또는 B의 병합 모델(없으면 새로 학습 30분) |

모든 평가에 `--confidence`를 붙여 모델의 토큰 로그 확률을 함께 저장합니다. 돌려받으면 위험·응답률 곡선(`scripts/risk_coverage.py`)을 EXAONE에 대해서도 그립니다.

## 0. 준비 (필수)

1번 노트북과 같습니다. Kaggle이면 Session options에서 Accelerator = *GPU T4 x2*, Internet = *On*, Add-ons → Secrets에 `HF_TOKEN`.

**결과를 잃지 않으려면 두 가지를 먼저 해 두세요.** Kaggle은 세션이 끝나거나 새로 시작되면 `/kaggle/working`을 비웁니다(소유자의 첫 실행 결과가 그렇게 사라졌습니다).

1. Session options → **Persistence = Files only**. 세션이 끊겨도 파일이 남습니다.
2. 절이 끝날 때마다 맨 아래에 뜨는 `gpu_results_2.tgz` 링크를 눌러 **그때그때 내려받습니다**. 각 절의 마지막 줄 `pack()`이 그때까지의 결과를 다시 묶습니다.

**Colab에서 돌릴 때.** 런타임 → 런타임 유형 변경 → *T4 GPU*. 왼쪽 열쇠 아이콘(보안 비밀)에 `HF_TOKEN`을 넣고 "노트북 액세스"를 켭니다. Colab은 런타임이 끊기면 `/content`가 지워지고 보존 설정이 없습니다. 그래서 아래 준비 셀이 Google Drive 연결을 묻습니다. 허용하면 절이 끝날 때마다 결과 묶음을 `내 드라이브/tabletrace_results/`에도 두고, 브라우저로도 바로 내려받습니다(브라우저가 "여러 파일 다운로드"를 물으면 허용). 런타임이 끊긴 뒤 다시 시작하면 Drive의 묶음을 되살려 끝난 평가를 건너뜁니다. GPU가 한 장이라 32B 모델(A, A2의 큰 모델)과 C절은 자동으로 건너뜁니다.

세션이 끊겼다면 이 절과 다음 셀을 다시 실행한 뒤 하던 절을 다시 실행하면 됩니다. **이미 끝난 학습과 평가는 건너뜁니다**(파일이 남아 있을 때). 학습은 끝에서만 저장하므로 도중에 멈춘 학습은 처음부터 다시 합니다. 어떤 결과를 일부러 다시 만들려면 그 폴더(`experiments/...`)를 지우고 실행하세요.

In [2]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import os, subprocess, time, json, glob, shutil
import torch
N_GPU = torch.cuda.device_count()
print("GPU 수:", N_GPU, "| compute capability:", torch.cuda.get_device_capability(0))
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        if not os.environ.get("HF_TOKEN"):
            print("HF_TOKEN이 없습니다. 공개 모델만 받을 수 있습니다.")

REPO = "https://github.com/sokldjs554/tabletrace"
if os.path.isdir("/kaggle/working"):
    ROOT, VENV = "/kaggle/working", "/tmp"
elif os.path.isdir("/content"):
    ROOT = VENV = "/content"
else:
    ROOT = VENV = os.path.join(os.path.expanduser("~"), "work"); os.makedirs(ROOT, exist_ok=True)
BACKUP = None                      # 결과 묶음을 한 벌 더 둘 곳. Colab은 런타임이 끊기면 /content가 지워지므로 Google Drive에 둔다
if ROOT == "/content":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BACKUP = "/content/drive/MyDrive/tabletrace_results"
        os.makedirs(BACKUP, exist_ok=True)
    except Exception as e:
        print("Google Drive를 연결하지 못했습니다. 결과는 브라우저 내려받기로만 남습니다:", e)
if not os.path.isdir(f"{ROOT}/tabletrace"):
    subprocess.run(["git", "clone", "-q", REPO, f"{ROOT}/tabletrace"], check=True)
REPO_DIR = f"{ROOT}/tabletrace"
os.chdir(REPO_DIR)
!git pull -q || echo "git pull 생략"
!pip -q install -e ".[dev]" peft accelerate sentencepiece
!pip -q uninstall -y torchao
!pip -q install uv
if not os.path.isdir(f"{VENV}/venv-vllm"):
    !uv venv --clear {VENV}/venv-vllm -q
    !uv pip install -q --python {VENV}/venv-vllm/bin/python vllm bitsandbytes
print("저장소:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
GPU 수: 2 | compute capability: (7, 5)
HF_TOKEN이 없습니다. 공개 모델만 받을 수 있습니다.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for tabletrace (pyproject.toml) ... done
저장소: b99ad95 Notebook 2: choose the download method by platform path, not by import


In [7]:
import requests

MODEL = "LGAI-EXAONE/EXAONE-4.0-1.2B"
GPU_NAME = torch.cuda.get_device_name(0).replace(" ", "_")
KO_TRAIN = "data/ko/train.jsonl data/ko/synthetic_train.jsonl"
KO_SETS = (("data/ko/test.jsonl data/ko/synthetic_test.jsonl", "ko_test"), ("data/ko/custom_eval.jsonl", "ko_custom"))

def stop_servers():
    """남아 있는 vLLM 서버는 GPU 메모리와 포트를 쥐고 있다. 다음 서버를 띄우기 전에 모두 끈다."""
    for sig in ("-TERM", "-KILL"):
        for pat in ("vllm serve", "vllm.entrypoints", "VLLM::"):
            subprocess.run(["pkill", sig, "-f", pat], capture_output=True)
        time.sleep(6)
    print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

def serve(model, port=8001, num_gpus=1, max_len=2048, dtype="half", extra="", gpu_util=0.88, log=None):
    """vLLM 서버를 띄우고 준비될 때까지 기다린다. 실패하면 로그 마지막 80줄을 보여 준다."""
    stop_servers()
    log = log or f"{ROOT}/vllm_{port}.log"
    e = dict(os.environ, PORT=str(port), PATH=f"{VENV}/venv-vllm/bin:" + os.environ["PATH"], PYTHONPATH="", PYTHONNOUSERSITE="1",
             MODEL=model, NUM_GPUS=str(num_gpus), DTYPE=dtype, MAX_LEN=str(max_len), GPU_UTIL=str(gpu_util), EXTRA_ARGS=extra)
    proc = subprocess.Popen(["bash", "scripts/serving/vllm_tp2.sh"], env=e, stdout=open(log, "w"), stderr=subprocess.STDOUT, cwd=REPO_DIR)
    for _ in range(360):
        try:
            requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=2).raise_for_status()
            return proc
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(5)
    raise RuntimeError("서버가 뜨지 않았습니다. 서버 로그 마지막 80줄:\n" + "".join(open(log).readlines()[-80:]))

def evaluated(name, sets=KO_SETS):
    """평가가 끝났는가. 세트마다 맨 마지막에 쓰는 파일(<...>_ec/metrics.json)이 있는지로 본다."""
    return all(os.path.isfile(f"experiments/eval/{name}_{suffix}_ec/metrics.json") for _, suffix in sets)

def trained(d, merge=True):
    """학습이 끝났는가. 학습 스크립트는 병합까지 마친 뒤 맨 끝에 train_info.json을 쓴다. 병합 모델이 필요한데 지워졌으면 다시 학습한다."""
    return os.path.isfile(f"{d}/train_info.json") and (not merge or os.path.isfile(f"{d}/merged/config.json"))

def free_weights(d):
    """평가가 끝난 모델의 병합 가중치(약 2.5GB)를 지운다. Kaggle 작업 폴더는 20GB까지라 모델을 다 두면 넘친다. 어댑터와 기록은 남는다."""
    shutil.rmtree(f"{d}/merged", ignore_errors=True)
    print("가중치 정리:", f"{d}/merged", "| 남은 디스크", round(shutil.disk_usage(ROOT).free / 2**30, 1), "GB")

def evaluate(name, tokenizer, sets=KO_SETS, corpus="data/ko/documents.jsonl", max_in=672, table_format="plain", port=8001, concurrency=16, extra=""):
    """같은 검증기·같은 지표로 평가한다. 결과: experiments/eval/<name>_<세트> (검증기 v1)와 <...>_ec (주어 검사 포함 재검증)."""
    for data, suffix in sets:
        out = f"experiments/eval/{name}_{suffix}"
        if os.path.isfile(f"{out}_ec/metrics.json"):
            print("건너뜀(이미 평가함):", out)
            continue
        common = f"--label {GPU_NAME} --tokenizer {tokenizer} --max-in {max_in} --table-format {table_format} --corpus {corpus} --data {data}"
        !python -m tabletrace.eval.run --planner engine --engine-url http://127.0.0.1:{port} --engine-model planner --concurrency {concurrency} --confidence {extra} {common} --out {out} --no-entity-check
        !python -m tabletrace.eval.run --planner engine --replay-from {out} {common} --out {out}_ec

def bench(name, tokenizer, port=8001, table_format="plain"):
    for c in (1, 8, 32):
        !python -m tabletrace.bench.engine_bench --url http://127.0.0.1:{port} --model planner --n 64 --concurrency {c} --tokenizer {tokenizer} --table-format {table_format} --out experiments/bench/engine_{name}_c{c}.json

def train_sft(out, limit=0, extra_n=None, table_format="plain", max_len=768, seed=0, train=KO_TRAIN, corpus="data/ko/documents.jsonl", ddp=False, more="", merge=True):
    """LoRA 튜닝. extra_n: 추가 학습 풀(data/ko/pool_train.jsonl.gz)에서 쓸 문항 수(None: 안 씀, 0: 전부). ddp=True면 GPU 2장으로 나눠 학습."""
    pool = "" if extra_n is None else f"--extra-train data/ko/pool_train.jsonl.gz --extra-n {extra_n}"
    corp = corpus + (" data/ko/pool_documents.jsonl.gz" if extra_n is not None else "")
    run = "torchrun --nproc_per_node 2" if ddp else "CUDA_VISIBLE_DEVICES=0 python"
    if trained(out, merge):
        print("건너뜀(이미 학습함):", out)
    else:
        merge_flag = "--merge" if merge else ""
        !{run} scripts/train/exaone_lora_sft.py --model {MODEL} --dtype auto --grad-ckpt {merge_flag} --train {train} {pool} --corpus {corp} \
            --epochs 1 --bs 2 --grad-accum 8 --limit {limit} --max-len {max_len} --table-format {table_format} --seed {seed} {more} --out {out}
    t = json.load(open(f"{out}/train_info.json"))
    print("학습", round(t.get("wall_s", 0) / 60, 1), "분 |", t["n_examples"], "예제 |", t["steps"], "step | GPU 프로세스", t.get("world_size", 1))

RESULT_PATTERNS = (
    "experiments/eval/ref_*", "experiments/eval/tools_*", "experiments/eval/exaone_pool*", "experiments/eval/exaone_rlstart_*", "experiments/eval/exaone_grpo_*", "experiments/eval/exaone_q_*",
    "experiments/eval/exaone_colid_s*", "experiments/eval/exaone_full_colid_*", "experiments/eval/exaone_kochem_*", "experiments/eval/exaone_*_unseen_dev*", "experiments/bench/engine_q_*",
    "experiments/exaone_lora_*/train_info.json", "experiments/exaone_grpo/train_info.json", "experiments/exaone_grpo/train_log.jsonl", "experiments/run_info_2.json")

def pack():
    """지금까지 만든 결과를 gpu_results_2.tgz로 묶는다(모델 가중치는 담지 않는다). 절이 끝날 때마다 부른다."""
    json.dump({"notebook": "gpu_experiments_2", "platform": "kaggle" if os.path.isdir("/kaggle/working") else "colab",
               "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)], "commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
               "finished": time.strftime("%Y-%m-%dT%H:%M:%S")}, open("experiments/run_info_2.json", "w"), indent=1)
    paths = [p for pat in RESULT_PATTERNS for p in sorted(glob.glob(pat))]
    out = f"{ROOT}/gpu_results_2.tgz"
    r = subprocess.run(["tar", "czf", out, *paths], capture_output=True, text=True)
    print("결과 묶음:", len(paths), "개 |", r.stderr.strip() or "tar ok", "|", out, os.path.getsize(out), "bytes | 남은 디스크", round(shutil.disk_usage(ROOT).free / 2**30, 1), "GB | 지금 내려받아 두세요")
    if BACKUP:
        try:
            shutil.copy(out, BACKUP)
            print("백업:", f"{BACKUP}/gpu_results_2.tgz")
        except Exception as e:
            print("백업하지 못했습니다:", e)
    try:
        from google.colab import files       # Colab: 브라우저로 바로 내려받는다
        files.download(out)
        return
    except ImportError:
        pass
    except Exception as e:
        print("자동 내려받기가 되지 않았습니다. 왼쪽 파일 패널에서 받으세요:", e)
        return
    try:
        from IPython.display import FileLink, display
        os.chdir(ROOT)                       # 링크는 현재 폴더 기준이라 잠시 옮긴다
        display(FileLink("gpu_results_2.tgz"))
    except Exception as e:
        print("내려받기 링크를 만들지 못했습니다. 오른쪽 Output 패널을 새로 고쳐 받으세요:", e)
    finally:
        os.chdir(REPO_DIR)

def restore_backup():
    """백업에 이전 세션의 결과 묶음이 있고 이 작업 폴더에 결과가 아직 없으면 풀어 놓는다. 끝난 평가를 건너뛰게 하려는 것이다.
    묶음에 모델 가중치는 없으므로, 아직 평가하지 않은 모델은 다시 학습한다."""
    src = f"{BACKUP}/gpu_results_2.tgz" if BACKUP else None
    if src and os.path.isfile(src) and not os.path.isfile("experiments/run_info_2.json"):
        subprocess.run(["tar", "xzf", src, "-C", REPO_DIR], check=True)
        print("이전 결과를 되살렸습니다:", src)

def done_tags(prefix, suffix, sets):
    """결과 이름이 <prefix><tag><suffix>인 평가 가운데 끝난 것의 tag. E, H절은 그때 있는 모델로 tag를 정하므로, 다른 세션에서 끝낸 것을 알아보는 데 쓴다."""
    last = sets[-1][1]
    tags = [os.path.basename(p)[len(prefix):-len(f"{suffix}_{last}_ec")] for p in glob.glob(f"experiments/eval/{prefix}*{suffix}_{last}_ec")]
    return sorted(t for t in tags if evaluated(f"{prefix}{t}{suffix}", sets))

restore_backup()

print("준비 끝. GPU:", GPU_NAME, "x", N_GPU)

준비 끝. GPU: Tesla_T4 x 2


## A. 큰 EXAONE을 튜닝 없이 (참고선)

이 프로젝트는 1.2B 모델만 튜닝했습니다. 여기서는 **튜닝하지 않은** EXAONE에 예시 6개(조회 2, 계산 2, 보류 2)를 보여 주고 같은 문법으로 답하게 해서, 같은 검증기로 채점합니다. 볼 것은 두 가지입니다.

- 모델이 26배 커지면(1.2B → 32B) 튜닝 없이도 **근거 셀**이 맞는가.
- 1.2B를 3.6천 문항으로 튜닝한 것(정답 중 근거 오류 1.3%)과 비교해 어느 쪽이 나은가.

순서: 1.2B(기준) → 7.8B AWQ(4비트, T4 한 장) → 32B AWQ(T4 두 장, 텐서 병렬). 32B는 메모리가 빠듯합니다. 서버가 뜨지 않으면 로그를 알려 주고 건너뛰세요.

In [8]:
SHOTS = f"--few-shot {KO_TRAIN} --shots-per-kind 2"
TOK = MODEL            # 행 예산을 튜닝 모델과 같게 맞추기 위해 1.2B 토크나이저로 센다
REFS = [
    ("ref_1.2b", "LGAI-EXAONE/EXAONE-4.0-1.2B", dict(num_gpus=1, max_len=6144)),
    ("ref_7.8b_awq", "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ", dict(num_gpus=1, max_len=6144, dtype="auto", extra="--trust-remote-code")),
]
if N_GPU >= 2:
    REFS.append(("ref_32b_awq", "LGAI-EXAONE/EXAONE-4.0-32B-AWQ", dict(num_gpus=2, max_len=6144, dtype="auto", gpu_util=0.95)))
for name, model, opts in REFS:
    if evaluated(name):
        print("건너뜀(이미 끝남):", name)
        continue
    try:
        proc = serve(model, **opts)
    except RuntimeError as e:
        print("건너뜀:", name, "\n", str(e)[-3000:])
        continue
    !nvidia-smi --query-gpu=index,memory.used --format=csv,noheader
    evaluate(name, TOK, extra=SHOTS, concurrency=8)
    proc.terminate(); proc.wait()
pack()

건너뜀(이미 끝남): ref_1.2b
0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
0, 13159 MiB
1, 3 MiB
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/tabletrace/src/tabletrace/eval/run.py", line 256, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/tabletrace/src/tabletrace/eval/run.py", line 196, in main
    res = resolve(e.question, e.table, prog, context=e.context, entity_check=not a.no_entity_check, column_check=a.column_check, **verify_opts)
  File "/kaggle/working/tabletrace/src/tabletrace/verify/verifier.py", line 208, in resolve
    result = decide()
  File "/kaggle/working/tabletrace/src/tabletrace/verify/verifier.py", line 183, in decide
    found = REGISTRY.call("cell_find", table, {"text": program.answer, "exact": True}, trace)
  File "/kaggle/working/tabletrace/src/tabletrace/tools/registry.py", line 75, in call
    raise ToolError(f"invalid arguments for {name}: {e.er

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## A2. 모델이 도구를 고르는 방식 (Tool Calling)

지금까지의 에이전트는 순서가 정해져 있습니다. 모델이 프로그램 한 줄을 쓰면 검증기가 도구를 실행합니다. 여기서는 **모델이 표 본문을 보지 않고**, 제목·열 이름·행 수만 받은 뒤 `cell_find`, `row_read`, `column_read`, `calculate` 도구를 스스로 골라 호출하고 `submit` 또는 `abstain`으로 끝냅니다(`src/tabletrace/agent/toolcall.py`). 제출한 프로그램은 같은 검증기를 거칩니다. 표를 도구로 읽으므로 입력 길이 제한이 없습니다.

튜닝하지 않은 EXAONE 4.0으로 시험합니다. vLLM은 도구 호출 해석을 켜고 띄웁니다(`--enable-auto-tool-choice --tool-call-parser hermes`). 서버가 이 옵션을 거부하면 로그를 알려 주세요. A절의 "예시를 보여 주고 한 줄로 답하기"와 같은 문항으로 비교됩니다. 큰 표(H절의 3,072문항 가운데 500개)도 함께 봅니다.

In [9]:
TOOL_ARGS = "--enable-auto-tool-choice --tool-call-parser hermes"
TOOL_REFS = [("tools_1.2b", "LGAI-EXAONE/EXAONE-4.0-1.2B", dict(num_gpus=1, max_len=8192, extra=TOOL_ARGS))]
if N_GPU >= 2:
    TOOL_REFS.append(("tools_32b_awq", "LGAI-EXAONE/EXAONE-4.0-32B-AWQ", dict(num_gpus=2, max_len=8192, dtype="auto", gpu_util=0.95, extra=TOOL_ARGS)))
UNSEEN500 = (("data/ko/unseen_dev.jsonl", "unseen_dev500"),)
for name, model, opts in TOOL_REFS:
    if evaluated(name) and evaluated(name, UNSEEN500):
        print("건너뜀(이미 끝남):", name)
        continue
    try:
        proc = serve(model, **opts)
    except RuntimeError as e:
        print("건너뜀:", name, "\n", str(e)[-3000:]); continue
    evaluate(name, TOK, extra="--tool-calling", concurrency=4)
    evaluate(name, TOK, sets=UNSEEN500, corpus="data/ko/unseen_dev_documents.jsonl.gz", extra="--tool-calling --sample 500", concurrency=4)
    proc.terminate(); proc.wait()
pack()

0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
{
 "meta": {
  "planner": "engine",
  "model": "engine:planner+tools",
  "data": [
   "data/ko/test.jsonl",
   "data/ko/synthetic_test.jsonl"
  ],
  "corpus": [
   "data/ko/documents.jsonl"
  ],
  "n": 699,
  "wall_s": 448.3,
  "hardware": "x86_64",
  "label": "Tesla_T4",
  "table_format": "plain",
  "row_select": "first",
  "evidence_repair": true,
  "entity_check": false,
  "column_check": false,
  "replay_from": null,
  "finished": "2026-11-14T15:14:37",
  "quantize": false,
  "quantize_scope": null,
  "confidence": false,
  "few_shot": null,
  "sample": null,
  "tool_calling": true
 },
 "metrics": {
  "n": 699,
  "n_answerable": 649,
  "n_gold_abstain": 50,
  "n_calc": 110,
  "n_lookup": 539,
  "verified_em": 0.1445,
  "answer_em_answerable": 0.0924,
  "raw_answer_em_answerable": 0.0971,
  "evidence_cells_exact": 0.0894,
  "evidence_cells_exact_raw": 0.1032,
  "evidence_repaired_rate": 0.0092,
  "evidence_cells_overlap": 0.0971,
  "op_acc": 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## B. 학습 데이터 확대

지금까지의 "전체 데이터" 모델은 조회 2,694문항과 합성 문항으로 학습했습니다. 표 크기 조건을 통과한 표 7,294개 가운데 900개만 썼기 때문입니다. 남은 표 6,111개의 질문 29,907개를 `data/ko/pool_train.jsonl.gz`에 모아 두었습니다(시험·개발 표와 같은 위키 문서에 있는 표는 뺐습니다).

- `EXTRA_N = 7000`: 약 1만 문항, T4 한 장에서 1.5시간 안팎(1번 노트북 실측 0.52초/문항 기준). GPU 두 장이면 `DDP = True`로 절반.
- `EXTRA_N = 0`: 풀 전부(약 3.4만 문항), 한 장 5시간, 두 장 2.5시간. 세션 12시간 제한 안입니다.

먼저 7000을 하고, 시간이 되면 0을 하세요. 결과 이름이 달라 서로 덮어쓰지 않습니다.

In [10]:
EXTRA_N = 7000          # 0이면 풀 전부
DDP = N_GPU >= 2
POOL_TAG = f"pool{EXTRA_N or 'all'}"
POOL_DIR = f"experiments/exaone_lora_{POOL_TAG}"
if evaluated(f"exaone_{POOL_TAG}_vllm"):
    print("건너뜀(이미 끝남):", POOL_TAG)
else:
    train_sft(POOL_DIR, extra_n=EXTRA_N, ddp=DDP)
    proc = serve(f"{REPO_DIR}/{POOL_DIR}/merged")
    evaluate(f"exaone_{POOL_TAG}_vllm", f"{REPO_DIR}/{POOL_DIR}/merged")
    proc.terminate(); proc.wait()
pack()                     # 이 모델의 가중치는 E, H절이 쓰므로 지우지 않는다

W1114 16:43:28.107000 3003 torch/distributed/run.py:851] 
W1114 16:43:28.107000 3003 torch/distributed/run.py:851] *****************************************
W1114 16:43:28.107000 3003 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1114 16:43:28.107000 3003 torch/distributed/run.py:851] *****************************************
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████| 332/332 [00:09<00:00, 35.72it/s]

trainable params: 6,389,760 || all params: 1,285,781,248 || trainable%: 0.4970
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setti

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## C. 멀티 GPU 학습 속도 (GPU 2장)

같은 1,000문항을 GPU 한 장과 두 장(DDP, `torchrun --nproc_per_node 2`)으로 학습해 걸린 시간을 비교합니다. 두 장일 때는 프로세스마다 데이터의 절반을 맡고 기울기를 평균합니다. 모델 품질 비교가 아니라 **속도 측정**입니다.

In [11]:
if N_GPU < 2:
    print("GPU가 1장이라 건너뜁니다.")
else:
    train_sft("experiments/exaone_lora_ddp1", limit=1000, more="--loop builtin", merge=False)      # 속도만 재므로 병합 모델을 만들지 않는다
    train_sft("experiments/exaone_lora_ddp2", limit=1000, ddp=True, merge=False)
    a, b = (json.load(open(f"experiments/exaone_lora_ddp{k}/train_info.json")) for k in (1, 2))
    print(f"1장 {a['wall_s']:.0f}초 ({a['steps']} step) | 2장 {b['wall_s']:.0f}초 ({b['steps']} step) | 속도 {a['wall_s'] / b['wall_s']:.2f}배")
    pack()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 332/332 [00:01<00:00, 321.82it/s]
trainable params: 6,389,760 || all params: 1,285,781,248 || trainable%: 0.4970
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 10, 'of': 62, 'loss': 0.057}
{'step': 20, 'of': 62, 'loss': 0.0956}
{'step': 30, 'of': 62, 'loss': 0.0676}
{'step': 40, 'of': 62, 'loss': 0.0825}
{'step': 50, 'of': 62, 'loss': 0.0142}
{'step': 60, 'of': 62, 'loss': 0.0194}
{'step': 62, 'of': 62, 'loss': 0.1732}
학습 8.7 분 | 1000 예제 | 62 step | GPU 프로세스 1
W1114 17:49:46.963000 3349 torch/distributed/run.py:851] 
W1114 17:49:46.963000 3349 torch/distributed/run.py:851] *****************************************
W1114 17:49:46.963000 3349 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune th

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## D. 강화학습 (GRPO) — 본인이 쓴 보상·손실 코드가 필요합니다

`docs/rewrite/06_training_code.md`의 A, B(`training/evidence_reward.py`, `training/grpo.py`)를 써서 `main`에 병합한 뒤 실행하세요. 없으면 첫 줄에서 안내와 함께 멈춥니다.

- 시작 모델: 300문항으로 튜닝한 모델. 정답 중 근거 셀 오류가 절반을 넘는 모델이라 고칠 것이 많습니다.
- 보상: 검증기가 근거 복구 없이 통과시킨 정답 +1.0, 복구가 있어야 정답 +0.3, 보류 0, 틀린 답 −0.5, 문법 오류 −1.0.
- 한 걸음: 프롬프트 4개 × 샘플 8개. 200걸음에 1~1.5시간 예상(실측 없음).
- 볼 것: 강화학습 전후의 **복구를 끈 정답률**과 **근거 셀 오류율**. 좋아지지 않아도 그대로 기록합니다.

In [ ]:
# 파일로 확인한다. 이 노트북의 커널은 패키지를 설치하기 전에 떠서 tabletrace를 import 할 수 없다(스크립트는 새 프로세스라 된다).
if not all(os.path.isfile(f"{REPO_DIR}/src/tabletrace/training/{m}.py") for m in ("evidence_reward", "grpo")):
    raise SystemExit("training/evidence_reward.py 와 training/grpo.py 가 없습니다. docs/rewrite/06_training_code.md 를 보고 직접 쓴 뒤 다시 실행하세요.")
START = "experiments/exaone_lora_rlstart"
if evaluated("exaone_rlstart_vllm") and os.path.isfile(f"{START}/adapter_config.json"):
    print("건너뜀(이미 끝남): exaone_rlstart_vllm")
else:
    train_sft(START, limit=300)
    proc = serve(f"{REPO_DIR}/{START}/merged")
    evaluate("exaone_rlstart_vllm", f"{REPO_DIR}/{START}/merged")          # 강화학습 전
    proc.terminate(); proc.wait()
stop_servers()
pack()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 332/332 [00:01<00:00, 308.15it/s]
trainable params: 6,389,760 || all params: 1,285,781,248 || trainable%: 0.4970
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 10, 'of': 18, 'loss': 0.2396}
{'step': 18, 'of': 18, 'loss': 0.3764}
Writing model shards: 100%|███████████████████████| 1/1 [00:03<00:00,  3.83s/it]
학습 2.6 분 | 300 예제 | 18 step | GPU 프로세스 1
0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
{
 "meta": {
  "planner": "engine",
  "model": "engine:planner",
  "data": [
   "data/ko/test.jsonl",
   "data/ko/synthetic_test.jsonl"
  ],
  "corpus": [
   "data/ko/documents.jsonl"
  ],
  "n": 699,
  "wall_s": 113.2,
  "hardware": "x86_64",
  "label": "Tesla_T4",
  "table_format": "plain",
  "row_select": "first",
  "evidence_repair": true,
  "entity_check": false,
  "column_check": false,
  "replay_from": null,
  "finished"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
GRPO_DIR = "experiments/exaone_grpo"      # T4 한 장 기준. --chunk 2 는 메모리가 넘쳤다(소유자 실행). 그래도 모자라면 --grad-ckpt 를 붙이거나 --group 4
if evaluated("exaone_grpo_vllm"):
    print("건너뜀(이미 끝남): exaone_grpo_vllm")
else:
    if os.path.isfile(f"{GRPO_DIR}/train_info.json") and os.path.isfile(f"{GRPO_DIR}/merged/config.json"):
        print("건너뜀(이미 학습함):", GRPO_DIR)
    else:
        !CUDA_VISIBLE_DEVICES=0 python scripts/train/exaone_grpo.py --model {MODEL} --adapter {START} --dtype auto \
            --train {KO_TRAIN} --dev data/ko/dev.jsonl data/ko/synthetic_dev.jsonl --corpus data/ko/documents.jsonl \
            --steps 200 --prompts 4 --group 8 --chunk 1 --lr 2e-5 --eval-every 25 --dev-n 64 --merge --out {GRPO_DIR}
    t = json.load(open(f"{GRPO_DIR}/train_info.json"))
    print("갱신", t["updates_applied"], "/", t["steps"], "| 고른 걸음", t["kept_step"], "| 개발셋 기록", [(h["step"], h["mean_reward"], h["correct"]) for h in t["dev_history"]])
    proc = serve(f"{REPO_DIR}/{GRPO_DIR}/merged")
    evaluate("exaone_grpo_vllm", f"{REPO_DIR}/{GRPO_DIR}/merged")           # 강화학습 후
    proc.terminate(); proc.wait()
    free_weights(GRPO_DIR)
pack()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 332/332 [00:01<00:00, 298.39it/s]
DEV {'step': 0, 'n': 64, 'mean_reward': 0.0172, 'correct': 16, 'repaired': 17, 'withheld': 7, 'wrong': 8, 'unparsable': 16}
{'step': 1, 'reward_mean': -0.2375, 'groups_all_equal': 0.25, 'pg_loss': 0.03273, 'kl': 0.0, 'grad_norm': None, 'update_applied': False, 'correct': 2, 'repaired': 3, 'withheld': 14, 'wrong': 5, 'unparsable': 8, 'minutes': 0.36}
{'step': 2, 'reward_mean': 0.0125, 'groups_all_equal': 0.0, 'pg_loss': -0.0062, 'kl': 0.0, 'grad_norm': None, 'update_applied': False, 'correct': 2, 'repaired': 8, 'withheld': 14, 'wrong': 8, 'unparsable': 0, 'minutes': 0.63}
{'step': 3, 'reward_mean': -0.2969, 'groups_all_equal': 0.5, 'pg_loss': 0.17849, 'kl': 0.0, 'grad_norm': 0.6444, 'update_applied': True, 'correct': 5, 'repaired': 0, 'withheld': 6, 'wrong': 13, 'unparsable': 8, 'minutes': 0.96}
{'step': 4, 'reward_mean': -0.2875, 'groups_all_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## E. 4비트 양자화 (bitsandbytes)

튜닝한 모델을 vLLM이 4비트로 올리게 해서(`--quantization bitsandbytes`), fp16과 같은 문항으로 정확도·처리량·GPU 메모리를 비교합니다. `QMODEL`은 1번 노트북의 전체 데이터 모델이 이 세션에 없으면 B에서 만든 모델을 씁니다. 둘 다 없으면 300문항 모델을 새로 만듭니다.

In [ ]:
Q_DONE = [t for t in done_tags("exaone_q_fp16_", "_vllm", KO_SETS) if t in done_tags("exaone_q_bnb4_", "_vllm", KO_SETS)]
cands = [f"{REPO_DIR}/experiments/exaone_lora_full/merged"] + sorted(glob.glob(f"{REPO_DIR}/experiments/exaone_lora_pool*/merged")) + [f"{REPO_DIR}/experiments/exaone_lora_rlstart/merged"]
QMODEL = next((c for c in cands if os.path.isfile(f"{c}/config.json")), None)
if Q_DONE:
    print("건너뜀(이미 끝남): 양자화 비교", Q_DONE)
elif QMODEL is None:
    train_sft("experiments/exaone_lora_rlstart", limit=300); QMODEL = f"{REPO_DIR}/experiments/exaone_lora_rlstart/merged"
QTAG = "" if Q_DONE else QMODEL.split("/")[-2].replace("exaone_lora_", "")
print("양자화할 모델:", None if Q_DONE else QMODEL)
for name, extra in (() if Q_DONE else ((f"q_fp16_{QTAG}", ""), (f"q_bnb4_{QTAG}", "--quantization bitsandbytes"))):
    try:
        proc = serve(QMODEL, extra=extra, dtype="half" if not extra else "auto")
    except RuntimeError as e:
        print("건너뜀:", name, "\n", str(e)[-3000:]); continue
    mem = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split()
    json.dump({"model": QMODEL, "vllm_args": extra, "gpu_memory_used_mib": [int(x) for x in mem], "note": "vLLM은 gpu-memory-utilization만큼 미리 잡으므로 이 값은 모델 크기가 아니라 서버가 잡은 양이다"},
              open(f"experiments/bench/engine_{name}_memory.json", "w"), indent=1)
    evaluate(f"exaone_{name}_vllm", QMODEL)
    bench(name, QMODEL)
    proc.terminate(); proc.wait()
pack()

양자화할 모델: /kaggle/working/tabletrace/experiments/exaone_lora_pool7000/merged
0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
{
 "meta": {
  "planner": "engine",
  "model": "engine:planner",
  "data": [
   "data/ko/test.jsonl",
   "data/ko/synthetic_test.jsonl"
  ],
  "corpus": [
   "data/ko/documents.jsonl"
  ],
  "n": 699,
  "wall_s": 114.5,
  "hardware": "x86_64",
  "label": "Tesla_T4",
  "table_format": "plain",
  "row_select": "first",
  "evidence_repair": true,
  "entity_check": false,
  "column_check": false,
  "replay_from": null,
  "finished": "2026-11-14T19:20:47",
  "quantize": false,
  "quantize_scope": null,
  "confidence": true,
  "few_shot": null,
  "sample": null,
  "tool_calling": false
 },
 "metrics": {
  "n": 699,
  "n_answerable": 649,
  "n_gold_abstain": 50,
  "n_calc": 110,
  "n_lookup": 539,
  "verified_em": 0.8884,
  "answer_em_answerable": 0.8814,
  "raw_answer_em_answerable": 0.8398,
  "evidence_cells_exact": 0.8798,
  "evidence_cells_exact_raw": 0.8891,
  "evidence_rep

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## F. 열 번호 형식 — 전체 데이터와 시드 추가

1번 노트북에서 300문항·시드 하나로 본 결과(근거 셀 오류 53~65% → 3.2%)를 넓힙니다. (1) 300문항을 시드 1, 2로 두 번 더, (2) 전체 데이터에서 한 번. 전체 데이터의 `plain` 근거 오류는 이미 1.3%라 차이가 작을 것으로 예상합니다. 그 예상이 맞는지 보는 실험입니다.

In [ ]:
runs = [(f"experiments/exaone_lora_colid_s{seed}", f"exaone_colid_s{seed}_vllm", dict(limit=300, seed=seed)) for seed in (1, 2)]
runs.append(("experiments/exaone_lora_full_colid", "exaone_full_colid_vllm", dict()))
for d, name, kw in runs:
    if evaluated(name):
        print("건너뜀(이미 끝남):", name)
        continue
    train_sft(d, table_format="colid", max_len=896, **kw)
    proc = serve(f"{REPO_DIR}/{d}/merged")
    evaluate(name, f"{REPO_DIR}/{d}/merged", max_in=800, table_format="colid")
    proc.terminate(); proc.wait()
    free_weights(d)
    pack()
pack()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 332/332 [00:01<00:00, 305.03it/s]
trainable params: 6,389,760 || all params: 1,285,781,248 || trainable%: 0.4970
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 10, 'of': 18, 'loss': 0.1283}
{'step': 18, 'of': 18, 'loss': 0.0349}
Writing model shards: 100%|███████████████████████| 1/1 [00:03<00:00,  3.46s/it]
학습 3.2 분 | 300 예제 | 18 step | GPU 프로세스 1
0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
{
 "meta": {
  "planner": "engine",
  "model": "engine:planner",
  "data": [
   "data/ko/test.jsonl",
   "data/ko/synthetic_test.jsonl"
  ],
  "corpus": [
   "data/ko/documents.jsonl"
  ],
  "n": 699,
  "wall_s": 143.6,
  "hardware": "x86_64",
  "label": "Tesla_T4",
  "table_format": "colid",
  "row_select": "first",
  "evidence_repair": true,
  "entity_check": false,
  "column_check": false,
  "replay_from": null,
  "finished"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 332/332 [00:01<00:00, 280.39it/s]
trainable params: 6,389,760 || all params: 1,285,781,248 || trainable%: 0.4970
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 10, 'of': 18, 'loss': 0.1543}
{'step': 18, 'of': 18, 'loss': 0.1254}
Writing model shards: 100%|███████████████████████| 1/1 [00:03<00:00,  3.34s/it]
학습 3.1 분 | 300 예제 | 18 step | GPU 프로세스 1
0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
{
 "meta": {
  "planner": "engine",
  "model": "engine:planner",
  "data": [
   "data/ko/test.jsonl",
   "data/ko/synthetic_test.jsonl"
  ],
  "corpus": [
   "data/ko/documents.jsonl"
  ],
  "n": 699,
  "wall_s": 144.1,
  "hardware": "x86_64",
  "label": "Tesla_T4",
  "table_format": "colid",
  "row_select": "first",
  "evidence_repair": true,
  "entity_check": false,
  "column_check": false,
  "replay_from": null,
  "finished"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (2 outputs left out)

<IPython.core.display.Javascript object>

결과 묶음: 67 개 | tar ok | /kaggle/working/gpu_results_2.tgz 2181053 bytes | 남은 디스크 14.2 GB | 지금 내려받아 두세요


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## G. 영어 화학 문서

한국어 표와 화학 합성 문항(820개)을 함께 학습한 EXAONE을 화학 평가 세 세트로 채점합니다. 교과서 표의 수작업 32문항과 합성 개발 80문항, 그리고 **실제 화학 논문 12편(arXiv, CC BY 4.0)의 표 20개에 대한 수작업 62문항**(`data/chem_papers/`)입니다. 논문 표는 학습에 없던 것이고, 단위 행·수식 기호·불확도 표기가 섞여 있어 더 어렵습니다. mT5-small은 여기서 약했습니다(교과서 수작업 18.8%, 논문 표 19~23%, 휴리스틱은 43.5%). 같은 모델이 한국어 성능을 유지하는지도 함께 봅니다.

In [ ]:
CHEM_DIR = "experiments/exaone_lora_kochem"
CHEM_SETS = (("data/chem/custom_eval.jsonl", "chem_custom"), ("data/chem/synthetic_dev.jsonl", "chem_syndev"))
PAPER_SETS = (("data/chem_papers/custom_eval.jsonl", "chem_papers"),)
if evaluated("exaone_kochem_vllm", CHEM_SETS) and evaluated("exaone_kochem_vllm", PAPER_SETS) and evaluated("exaone_kochem_vllm"):
    print("건너뜀(이미 끝남): exaone_kochem_vllm")
else:
    train_sft(CHEM_DIR, train=KO_TRAIN + " data/chem/synthetic_train.jsonl", corpus="data/ko/documents.jsonl data/chem/documents.jsonl")
    proc = serve(f"{REPO_DIR}/{CHEM_DIR}/merged")
    evaluate("exaone_kochem_vllm", f"{REPO_DIR}/{CHEM_DIR}/merged", corpus="data/chem/documents.jsonl", sets=CHEM_SETS)
    evaluate("exaone_kochem_vllm", f"{REPO_DIR}/{CHEM_DIR}/merged", corpus="data/chem_papers/documents.jsonl",
             sets=PAPER_SETS, extra="--row-select overlap")      # 실제 화학 논문 12편의 표 62문항
    evaluate("exaone_kochem_vllm", f"{REPO_DIR}/{CHEM_DIR}/merged")       # 한국어 성능이 유지되는지
    proc.terminate(); proc.wait()
    free_weights(CHEM_DIR)
pack()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 332/332 [00:01<00:00, 313.36it/s]
trainable params: 6,389,760 || all params: 1,285,781,248 || trainable%: 0.4970
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 10, 'of': 273, 'loss': 0.389}
{'step': 20, 'of': 273, 'loss': 0.3095}
{'step': 30, 'of': 273, 'loss': 0.0458}
{'step': 40, 'of': 273, 'loss': 0.0182}
{'step': 50, 'of': 273, 'loss': 0.4531}
{'step': 60, 'of': 273, 'loss': 0.0225}
{'step': 70, 'of': 273, 'loss': 0.0182}
{'step': 80, 'of': 273, 'loss': 0.0011}
{'step': 90, 'of': 273, 'loss': 0.0095}
{'step': 100, 'of': 273, 'loss': 0.0243}
{'step': 110, 'of': 273, 'loss': 0.221}
{'step': 120, 'of': 273, 'loss': 0.0006}
{'step': 130, 'of': 273, 'loss': 0.0036}
{'step': 140, 'of': 273, 'loss': 0.0018}
{'step': 150, 'of': 273, 'loss': 0.0032}
{'step': 160, 'of': 273, 'loss': 0.2818}
{'step': 170, 'of': 273, '

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## H. 큰 표와 공개 dev 분할

지금까지의 시험 표는 16행·7열·900자 이하였습니다. `data/ko/unseen_dev.jsonl`은 **공개 dev 분할에서 학습에 쓰이지 않은 표의 질문 3,072개**이고 97%가 그 한도를 넘는 표입니다. 표가 입력에 다 들어가지 않을 때 앞에서부터 자르는 방식(`first`)과 질문과 겹치는 행을 고르는 방식(`overlap`, 행 번호는 원래 값 유지)을 비교합니다. 모델은 다시 학습하지 않고, 이 세션에 있는 튜닝 모델을 그대로 씁니다. 입력 예산을 1,500토큰으로 늘린 경우도 함께 봅니다(학습 때 본 길이는 672 이하).

In [ ]:
UNSEEN = (("data/ko/unseen_dev.jsonl", "unseen_dev"),)
H_RUNS = (("first", 672), ("overlap", 672), ("first", 1500))
H_DONE = [t for t in done_tags("exaone_", "_first672_vllm", UNSEEN) if all(evaluated(f"exaone_{t}_{rs}{max_in}_vllm", UNSEEN) for rs, max_in in H_RUNS)]
if H_DONE:
    print("건너뜀(이미 끝남): 큰 표 평가", H_DONE)
else:
    cands = [f"{REPO_DIR}/experiments/exaone_lora_full/merged"] + sorted(glob.glob(f"{REPO_DIR}/experiments/exaone_lora_pool*/merged")) + [f"{REPO_DIR}/experiments/exaone_lora_rlstart/merged"]
    LMODEL = next((c for c in cands if os.path.isfile(f"{c}/config.json")), None)
    if LMODEL is None:
        train_sft("experiments/exaone_lora_full"); LMODEL = f"{REPO_DIR}/experiments/exaone_lora_full/merged"
    LTAG = LMODEL.split("/")[-2].replace("exaone_lora_", "")
    print("평가할 모델:", LMODEL)
    proc = serve(LMODEL, max_len=4096)
    for rs, max_in in H_RUNS:
        evaluate(f"exaone_{LTAG}_{rs}{max_in}_vllm", LMODEL, sets=UNSEEN, corpus="data/ko/unseen_dev_documents.jsonl.gz", max_in=max_in, extra=f"--row-select {rs}")
    proc.terminate(); proc.wait()
pack()

평가할 모델: /kaggle/working/tabletrace/experiments/exaone_lora_pool7000/merged
0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
{
 "meta": {
  "planner": "engine",
  "model": "engine:planner",
  "data": [
   "data/ko/unseen_dev.jsonl"
  ],
  "corpus": [
   "data/ko/unseen_dev_documents.jsonl.gz"
  ],
  "n": 3072,
  "wall_s": 657.1,
  "hardware": "x86_64",
  "label": "Tesla_T4",
  "table_format": "plain",
  "row_select": "first",
  "evidence_repair": true,
  "entity_check": false,
  "column_check": false,
  "replay_from": null,
  "finished": "2026-11-14T21:19:29",
  "quantize": false,
  "quantize_scope": null,
  "confidence": true,
  "few_shot": null,
  "sample": null,
  "tool_calling": false
 },
 "metrics": {
  "n": 3072,
  "n_answerable": 3072,
  "n_gold_abstain": 0,
  "n_calc": 0,
  "n_lookup": 3072,
  "verified_em": 0.7262,
  "answer_em_answerable": 0.7262,
  "raw_answer_em_answerable": 0.7262,
  "evidence_cells_exact": 0.7259,
  "evidence_cells_exact_raw": 0.7454,
  "evidence_repaired_rate": 0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## I. 결과 돌려주기

아래 셀이 이 노트북에서 만든 결과만 묶습니다(모델 가중치는 담지 않습니다). 뜨는 링크를 눌러 `gpu_results_2.tgz`를 받아서 대화에 올려 주세요. 각 절의 끝에서도 같은 묶음을 만들기 때문에, 마지막에 받은 것 하나만 올리면 됩니다.

In [ ]:
stop_servers()
pack()

0, 3 MiB, 15360 MiB
1, 3 MiB, 15360 MiB
결과 묶음: 90 개 | tar ok | /kaggle/working/gpu_results_2.tgz 4683469 bytes | 남은 디스크 14.2 GB | 지금 내려받아 두세요


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from IPython.display import FileLink, display
os.chdir(ROOT); display(FileLink("gpu_results_2.tgz")); os.chdir(REPO_DIR)

/kaggle/working/gpu_results_2.tgz

In [4]:
import sys; sys.modules["google.colab"] = None